# 读取数据

##  振动cwt数据读取

In [2]:
import torch
from torch_geometric.data import Data

# ===== 原始路径 =====
paths = {
    "train": "/home/charles/HZU/Data_processed/my_CIL_V1/KAIST/KAIST_CWT/normalization/train/CWT_pyg/train_graph_with_labelmask.pt",
    "val"  : "/home/charles/HZU/Data_processed/my_CIL_V1/KAIST/KAIST_CWT/normalization/val/CWT_pyg/graph.pt",
    "test" : "/home/charles/HZU/Data_processed/my_CIL_V1/KAIST/KAIST_CWT/normalization/test/CWT_pyg/graph.pt",
}

def to_pyg_data(path):
    data = torch.load(path)

    pyg_data = Data(
        x=data.x,
        edge_index=data.edge_index,
        y=data.y if hasattr(data, "y") else None
    )

    # 保留所有 mask
    for attr in dir(data):
        if "mask" in attr:
            m = getattr(data, attr)
            if torch.is_tensor(m):
                setattr(pyg_data, attr, m)

    return pyg_data


# ===== 存到变量 =====
vibration_cwt_train = to_pyg_data(paths["train"])
vibration_cwt_val   = to_pyg_data(paths["val"])
vibration_cwt_test  = to_pyg_data(paths["test"])


# ===== 快速 sanity check =====
print("Train :", vibration_cwt_train)
print("Val   :", vibration_cwt_val)
print("Test  :", vibration_cwt_test)


Train : Data(x=[1639, 124], edge_index=[2, 32780], y=[1639])
Val   : Data(x=[272, 124], edge_index=[2, 5440], y=[272])
Test  : Data(x=[819, 124], edge_index=[2, 16380], y=[819])


/tmp/ipykernel_832654/883891547.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)


## 读取电流和温度数据

In [3]:
import torch
from torch_geometric.data import Data

# ===== 原始路径 =====
paths_ct = {
    "train": "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/train/pyg/train_graph_with_labelmask.pt",
    "val"  : "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/val/pyg/graph.pt",
    "test" : "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/test/pyg/graph.pt",
}

def to_pyg_data(path):
    data = torch.load(path)

    pyg_data = Data(
        x=data.x,
        edge_index=data.edge_index,
        y=data.y if hasattr(data, "y") else None
    )

    # 保留所有 mask（train_withlabel_mask / train_nolabel_mask 等）
    for attr in dir(data):
        if "mask" in attr:
            m = getattr(data, attr)
            if torch.is_tensor(m):
                setattr(pyg_data, attr, m)

    return pyg_data


# ===== 存为变量 =====
current_temp_train = to_pyg_data(paths_ct["train"])
current_temp_val   = to_pyg_data(paths_ct["val"])
current_temp_test  = to_pyg_data(paths_ct["test"])


# ===== sanity check =====
print("Train :", current_temp_train)
print("Val   :", current_temp_val)
print("Test  :", current_temp_test)


Train : Data(x=[45000, 5], edge_index=[2, 900000], y=[45000])
Val   : Data(x=[7500, 5], edge_index=[2, 150000], y=[7500])
Test  : Data(x=[22500, 5], edge_index=[2, 450000], y=[22500])


/tmp/ipykernel_832654/3887575172.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)


### 利用滑窗对齐图规模

In [5]:
import torch
from torch_geometric.data import Data

def window_align_graph(raw_graph, target_num_nodes):
    """
    raw_graph: PyG Data
        x: [N_raw, F]
        y: [N_raw]
    target_num_nodes: int (振动图该 split 的节点数)
    """
    x_raw = raw_graph.x
    y_raw = raw_graph.y

    N_raw = x_raw.size(0)
    F = x_raw.size(1)

    win_size = N_raw // target_num_nodes
    assert win_size > 0, f"Invalid window size: {win_size}"

    x_win, y_win = [], []

    for i in range(target_num_nodes):
        start = i * win_size
        end = (i + 1) * win_size if i < target_num_nodes - 1 else N_raw

        x_slice = x_raw[start:end]    # [w, F]
        y_slice = y_raw[start:end]    # [w]

        # ---- 特征：窗口均值 ----
        x_feat = x_slice.mean(dim=0)

        # ---- 标签：多数投票 ----
        labels, counts = torch.unique(y_slice, return_counts=True)
        y_label = labels[counts.argmax()]

        x_win.append(x_feat)
        y_win.append(y_label)

    x_win = torch.stack(x_win)   # [N_target, F]
    y_win = torch.stack(y_win)   # [N_target]

    return Data(x=x_win, y=y_win)


# =========================================================
# 对三个 split 分别对齐（窗口大小各不相同）
# =========================================================

# ---- 振动图节点数（目标）----
N_train = vibration_cwt_train.x.size(0)
N_val   = vibration_cwt_val.x.size(0)
N_test  = vibration_cwt_test.x.size(0)

# ---- 电流&温度对齐 ----
ct_train_aligned = window_align_graph(current_temp_train, N_train)
ct_val_aligned   = window_align_graph(current_temp_val,   N_val)
ct_test_aligned  = window_align_graph(current_temp_test,  N_test)

# ---- sanity check ----
print("Train aligned:", ct_train_aligned.x.shape, ct_train_aligned.y.shape)
print("Val aligned  :", ct_val_aligned.x.shape,   ct_val_aligned.y.shape)
print("Test aligned :", ct_test_aligned.x.shape,  ct_test_aligned.y.shape)


Train aligned: torch.Size([1639, 5]) torch.Size([1639])
Val aligned  : torch.Size([272, 5]) torch.Size([272])
Test aligned : torch.Size([819, 5]) torch.Size([819])


# GCL表征学习